# 🫁 CXR DenseNet121 — **v2 ÓPTIMO** (DenseNet121 congelado + cabeza MLP · CPU)
## TFM · Módulo Imagen (Radiografía de tórax) · Universidad de Salamanca

---

Versión de referencia del módulo de imagen: **DenseNet-121 de `torchxrayvision` (pesos CheXNet)
congelado** como extractor de embeddings, y una **cabeza MLP** entrenada encima, con metadatos
clínicos como rama auxiliar, tuning por **Optuna** y **K-fold** para OOF sin fuga.

## ⚠️ ESTE NOTEBOOK GENERA LOS EMBEDDINGS QUE USA v1
La extracción de embeddings (**~30–45 min en CPU**, una sola vez) se guarda en
`salidas/01_cxr/v2/embeddings/`. **`01_CXR_DenseNet121_v1` los lee de ahí y aborta si no existen.**
Por eso, en imagen, **v2 se ejecuta ANTES que v1**.

> **Corregido:** la ruta de salida apuntaba a una carpeta relativa antigua
> (`outputs_cxr_densenet121_v2_uignore/`) mientras que v1 buscaba en `salidas/01_cxr/v2/`.
> No coincidían, así que v1 habría fallado incluso después de ejecutar v2. Ahora ambas apuntan al
> mismo sitio.

## 🔄 Adaptación a las conclusiones del EDA

| Cambio | Detalle | Origen (recuadro naranja) |
|---|---|---|
| **Etiquetado FINAL** | `POS=(==1)` · `NEG=(==0)|(NaN→0)` · **−1 ENMASCARADO** (U-Ignore). **Se elimina** la derivación de negativos desde *No Finding* (`DERIVE_NEGATIVES_FROM_EXCLUSIVITY=False`). | §3 Definición del negativo |
| **Métrica primaria = AUC-PR** | `macro_AP_path` en Optuna, early-stopping, K-fold y selección; **AUC-ROC secundaria**; **IC bootstrap** de AP y AUC. | §3 Métrica · §1 IC |
| **Puntos de operación** | F1, **cribado (Se≥0,90)** y **confirmación (Sp≥0,90)** fijados en VAL, con **Se/Sp/VPP/VPN**. | §11 Puntos de operación |
| **Calibración verificada** | **Brier** antes/después + curva de fiabilidad. | §11 Calibración |
| **Pre-registro** | Regla de decisión congelada en JSON **antes** de tocar test. | §11 Protocolo |
| **Control de calidad de imagen** | Chequeo de contraste mínimo (std de píxel) sobre una muestra, **antes** de extraer embeddings. | Anexo EDA (señal de imagen) |
| **Equidad y robustez** | Estratificación por sexo, etnia, ingreso y **proyección AP/PA**. | §2/§9 · Anexo CXR |

### Sobre la normalización de la imagen (matiz importante)
Los `.npy` vienen **z-scored al estilo ImageNet** (3 canales, ~[−2,1 , 2,6]), como documentó el anexo
del EDA. Pero este notebook usa **`torchxrayvision`**, cuyo backbone espera **1 canal en el rango
[−1024, 1024]**. La celda de preprocesado hace un **min-max por imagen** (el z-score es una
transformación afín, así que se recupera la estructura [0,1]) y luego reescala al rango esperado.

> **No es una contradicción con el anexo ni con la regla de "no re-normalizar":** el tensor no se
> altera arbitrariamente, se **adapta al rango que exige este backbone concreto**. Cambiarlo rompería
> el modelo. Se deja tal cual y se documenta.

- **`cxr_view` NUNCA como predictor**: la proyección AP se hace al paciente encamado (proxy de
  gravedad) y además la propia imagen ya la contiene. Solo se usa para **estratificar**.
- **B5 y B7 no aplican**: sin variables tabulares ni *missingness* que vigilar. → §6

> **Aviso:** con el nuevo etiquetado y la nueva métrica, los resultados **no son comparables** con los
> de la ejecución previa, y **los embeddings deben regenerarse**.



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 1 · INSTALACIÓN DE DEPENDENCIAS                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# torchxrayvision  -> backbone DenseNet121 preentrenado en radiografías
# optuna           -> búsqueda de hiperparámetros (TPE) sobre embeddings cacheados
# scikit-learn     -> métricas (AUC, AP, F1) + IsotonicRegression (calibración)
import subprocess, sys

packages = [
    "torchxrayvision", "optuna", "scikit-learn",
    "pandas", "numpy", "matplotlib", "seaborn", "tqdm"
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q", "--no-cache-dir"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"✅ {pkg}")
    else:
        print(f"⚠️  {pkg} — warning (puede estar ya instalado):")
        print(result.stderr[-300:])

# Verificar imports críticos
import importlib
critical = {"torchxrayvision": "torchxrayvision", "optuna": "optuna",
            "sklearn": "scikit-learn", "torch": "torch"}
print("\n── Verificación de imports ──")
for mod, pkg in critical.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        print(f"  ✅ {pkg} {ver}")
    except ImportError:
        print(f"  ❌ {pkg} — NO encontrado, instalar manualmente")

print("\nDependencias listas.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 2 · IMPORTS, SEMILLAS, HILOS DE CPU Y DISPOSITIVO                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os, gc, json, time, copy, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset, WeightedRandomSampler

import torchxrayvision as xrv

from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.isotonic import IsotonicRegression

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # menos ruido en el log

warnings.filterwarnings("ignore")

# --- Reproducibilidad ---------------------------------------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- Hilos de CPU: tu i5-1135G7 tiene 4 núcleos físicos -----------------------
N_CORES = 4
torch.set_num_threads(N_CORES)
os.environ["OMP_NUM_THREADS"] = str(N_CORES)
os.environ["MKL_NUM_THREADS"] = str(N_CORES)

# --- Dispositivo: no hay GPU NVIDIA en este equipo -> CPU ---------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch        : {torch.__version__}")
print(f"torchxrayvision: {xrv.__version__}")
print(f"Dispositivo    : {DEVICE}  (CPU esperado en este equipo)")
print(f"Hilos CPU      : {torch.get_num_threads()}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 3 · RUTAS, ETIQUETAS Y CONSTANTES GLOBALES                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# NOTA de diseño:
#  - Forzamos cxr_<split>.npy (320x320). NO usamos cxr_train_224.npy porque está
#    corrupto (pickled, no carga con mmap). Esto sustituye al [FIX 24] de v4.
#  - test_clean.csv tiene 464 filas (el .npy tiene 4640): es INTENCIONAL. El estudio
#    Symile original es de retrieval (1 positivo + 9 distractores), por eso el test
#    etiquetado "limpio" es un subconjunto. El join se queda con esas 464. Correcto.

BASE = Path(r"C:\TFM\1.Opción - Symile Mimic\symile-mimic-a-multimodal-clinical-dataset-of-chest-x-rays-electrocardiograms-and-blood-labs-from-mimic-iv-1.0.0")

CSV_DIR   = BASE / "data_csv" / "clean"
TRAIN_CSV = CSV_DIR / "train_clean.csv"
VAL_CSV   = CSV_DIR / "val_clean.csv"
TEST_CSV  = CSV_DIR / "test_clean.csv"

NPY_DIR        = BASE / "data_npy"
CXR_TRAIN_NPY  = NPY_DIR / "train" / "cxr_train.npy"
CXR_VAL_NPY    = NPY_DIR / "val"   / "cxr_val.npy"
CXR_TEST_NPY   = NPY_DIR / "test"  / "cxr_test.npy"
HADM_TRAIN_NPY = NPY_DIR / "train" / "hadm_id_train.npy"
HADM_VAL_NPY   = NPY_DIR / "val"   / "hadm_id_val.npy"
HADM_TEST_NPY  = NPY_DIR / "test"  / "hadm_id_test.npy"

# RUTA ABSOLUTA: CXR v1 lee los embeddings de aqui (salidas/01_cxr/v2/embeddings).
# Antes apuntaba a una carpeta relativa antigua y v1 no los encontraba.
OUTPUT_DIR = Path("C:/TFM/1.Opción - Symile Mimic/tfm_multimodal_clinico/salidas/01_cxr/v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figuras"; FIG_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR = OUTPUT_DIR / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)

# --- Etiquetas ----------------------------------------------------------------
LABELS = ["Atelectasis", "Cardiomegaly", "Edema", "Lung Opacity", "No Finding", "Pleural Effusion"]
N_LABELS = len(LABELS)
NO_FINDING = "No Finding"
PATHOLOGY_LABELS = [l for l in LABELS if l != NO_FINDING]            # 5 patologías
CORE_LABELS      = ["Cardiomegaly", "Edema", "Pleural Effusion"]      # balance razonable

# --- Preprocesado de imagen ---------------------------------------------------
IMG_SIZE = 224           # nativo de densenet121-res224-all
XRV_RANGE = 1024.0       # torchxrayvision espera entrada en [-1024, 1024]
BACKBONE_WEIGHTS = "densenet121-res224-all"

# --- Regla de negativos derivados (la idea central de la v5) ------------------
# Si No Finding==1 -> patologías=0 ; si alguna patología==1 -> No Finding=0.
# Solo rellena NaN; nunca pisa valores explícitos. Ponlo en False para comparar.
DERIVE_NEGATIVES_FROM_EXCLUSIVITY = False   # <- etiquetado FINAL: no derivar de No Finding (sesgo de espectro)

# --- Metadatos clínicos (rama auxiliar de la cabeza) --------------------------
GENDER_MAP    = {0: 0, 1: 1, "0": 0, "1": 1, "M": 1, "F": 0}
RACE_MAP      = {"UNKNOWN": 0, "WHITE": 1, "BLACK": 2, "ASIAN": 3, "HISPANIC_LATINO": 4, "OTHER_KNOWN": 0}
ADMISSION_MAP = {"SCHEDULED": 0, "EMERGENCY": 1, "OBSERVATION": 2, "URGENT": 3}
ADM_LOC_MAP   = {"EMERGENCY_ROOM": 0, "REFERRAL": 1, "TRANSFER": 2, "INTRA_HOSPITAL": 3}
CXR_VIEW_MAP  = {"AP": 0, "PA": 1}
RACE_ONEHOT   = 5   # indices 0..4 (OTHER_KNOWN se agrupa en UNKNOWN)
# META = edad + sexo + raza(5) + tipo_ingreso(4) + lugar_ingreso(4) + horas_ingreso_a_cxr(1) + vista(2)
# cxr_view NO entra en META_DIM: es proxy de gravedad (placa portatil = paciente encamado) y
# la imagen ya contiene la proyeccion. Se usa SOLO para estratificar resultados (bloque B8).
META_DIM = 1 + 1 + RACE_ONEHOT + len(ADMISSION_MAP) + len(ADM_LOC_MAP) + 1  # = 16 (antes 18, con vista)

# --- Presupuesto de cómputo (ajustable) ---------------------------------------
OPTUNA_TRIALS   = 25     # búsqueda de hiperparámetros (barata sobre embeddings)
K_FOLDS         = 5      # K-fold final -> estimación honesta + OOF para stacking
TUNE_EPOCHS     = 20     # epochs por entrenamiento durante la búsqueda
TUNE_PATIENCE   = 5
FINAL_EPOCHS    = 40     # epochs del K-fold final
FINAL_PATIENCE  = 8
HEAD_BATCH      = 128    # batch para la cabeza (embeddings -> rápido y BN estable)
EMB_BATCH       = 16     # batch para la pasada del backbone (imágenes)

print("Constantes configuradas.")
print(f"  Backbone           : {BACKBONE_WEIGHTS}  @ {IMG_SIZE}x{IMG_SIZE}")
print(f"  Negativos derivados: {DERIVE_NEGATIVES_FROM_EXCLUSIVITY}")
print(f"  META_DIM           : {META_DIM}")
print(f"  Macro 'core'       : {CORE_LABELS}")
print(f"  Macro 'patologías' : {PATHOLOGY_LABELS}")




In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 4 · CARGA Y ALINEACIÓN CSV <-> NPY (join por hadm_id)                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Cada fila del CSV limpio se enlaza con su imagen en el .npy mediante hadm_id.
# El .npy de imágenes se abre con mmap_mode='r' (mapeado en disco, no a RAM).

def load_split(csv_path, cxr_npy_path, hadm_npy_path, name):
    print(f"\n-- Split '{name}' " + "-" * 40)
    df = pd.read_csv(csv_path, sep=";")
    print(f"   CSV filas         : {len(df):,}")

    hadm = np.load(hadm_npy_path, allow_pickle=True)              # hadm_id por índice del npy
    hadm_to_idx = {int(h): i for i, h in enumerate(hadm)}
    print(f"   npy hadm_id count : {len(hadm):,}")

    df["_npy_idx"] = df["hadm_id"].map(lambda h: hadm_to_idx.get(int(h), -1))
    n0 = len(df)
    df = df[df["_npy_idx"] >= 0].reset_index(drop=True)           # descarta sin imagen
    print(f"   Filas con imagen  : {len(df):,}  (descartadas: {n0 - len(df)})")

    cxr = np.load(cxr_npy_path, mmap_mode="r")
    print(f"   CXR npy           : shape={cxr.shape} dtype={cxr.dtype}")
    return df, cxr

df_train, cxr_train = load_split(TRAIN_CSV, CXR_TRAIN_NPY, HADM_TRAIN_NPY, "train")
df_val,   cxr_val   = load_split(VAL_CSV,   CXR_VAL_NPY,   HADM_VAL_NPY,   "val")
df_test,  cxr_test  = load_split(TEST_CSV,  CXR_TEST_NPY,  HADM_TEST_NPY,  "test")

print(f"\nResumen: train={len(df_train):,}  val={len(df_val):,}  test={len(df_test):,}")


In [ ]:
# CELDA 5 · OBJETIVOS — definición FINAL del negativo (U-IGNORE del −1)
# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · build_targets
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : convierte los cuatro estados de cada etiqueta (1 / 0 / −1 / NaN) en dos matrices:
#              `labels` (0-1) y `mask` (1 = cuenta en pérdida y métrica, 0 = se ignora).
# POR QUÉ    : fija la definición FINAL del negativo acordada con el tutor, IDÉNTICA en las tres
#              modalidades — requisito para que la fusión combine probabilidades coherentes.
#              POS = (==1) · NEG = (==0) | (NaN→0) · −1 = ENMASCARADO (U-Ignore de CheXpert).
# ENTRADAS   : df (split) · uncertainty_policy ("ignore") · derive (compatibilidad) · verbose
# SALIDAS    : (labels (N,6) float32, mask (N,6) float32)
# ORIGEN EDA : §3 · recuadro naranja "negativo = 0 explícito + NaN→0; el −1 se enmascara; «Sin
#              hallazgo» NO como negativo (evita el sesgo de espectro)".
# CAMBIO vs versión previa: se ELIMINA la derivación de negativos desde «No Finding»; el NaN pasa a
#              NEGATIVO con máscara activa en lugar de quedar enmascarado.
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              comprobar pos_weight ≈ 2,28 / 1,85 / 3,43 / 2,15 / 6,31 / 1,66 en train; advertir de que
#              las métricas NO son comparables con las de la ejecución anterior.
# ══════════════════════════════════════════════════════════════════════════════
def build_targets(df,uncertainty_policy="ignore",derive=DERIVE_NEGATIVES_FROM_EXCLUSIVITY,verbose=False):
    raw=df[LABELS].to_numpy(dtype=float); N=raw.shape[0]
    labels=(raw==1.0).astype(np.float32)          # 1 → positivo ; 0 y NaN → 0 (negativo)
    mask  =np.ones((N,N_LABELS),np.float32)       # por defecto TODO entra en pérdida/métrica
    unc=(raw==-1.0)
    if   uncertainty_policy=="ignore": mask[unc]=0.0    # −1 → ENMASCARADO (definición FINAL)
    elif uncertainty_policy=="ones":   labels[unc]=1.0  # alternativa, no usada
    n_dp=n_dn=0
    if derive:   # CONSERVADO por compatibilidad; NO forma parte de la definición FINAL
        nf=LABELS.index(NO_FINDING); pc=[j for j in range(N_LABELS) if j!=nf]; nfp=(raw[:,nf]==1)
        for j in pc:
            f=nfp&np.isnan(raw[:,j]); labels[f,j]=0.0; n_dp+=int(f.sum())
        ap=(raw[:,pc]==1).any(1); fn=ap&np.isnan(raw[:,nf]); labels[fn,nf]=0.0; n_dn=int(fn.sum())
    if verbose and derive: print(f"   Neg. derivados (OFF por defecto): patologias={n_dp:,} · No Finding={n_dn:,}")
    return labels,mask

print("== Balance train con el etiquetado FINAL (POS=1; NEG=0+NaN; -1 enmascarado) ==")
_y,_m=build_targets(df_train,"ignore",verbose=True)
for j,l in enumerate(LABELS):
    s=_m[:,j]==1; p=int((_y[s,j]==1).sum()); n=int((_y[s,j]==0).sum())
    print(f"   {l:18s} pos={p:5d} neg={n:5d} enmascarados={int((_m[:,j]==0).sum()):4d} pos_weight={n/max(p,1):.2f}")



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 6 · METADATOS CLÍNICOS (rama auxiliar)                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · build_metadata
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : construye un vector de 16 dimensiones por paciente: edad normalizada (1), sexo (1),
#              one-hot de raza (5), tipo de ingreso (4), lugar de ingreso (4) y horas desde el
#              ingreso hasta la radiografía (1).
# POR QUÉ    : da a la cabeza contexto clínico que la imagen sola no aporta. La EDAD es uno de los
#              dos predictores más fuertes del estudio, y el tipo/lugar de ingreso son proxies del
#              curso clínico.
# ENTRADAS   : df (DataFrame del split con las columnas demográficas)
# SALIDAS    : (N, META_DIM) float32
# ORIGEN EDA : §2 · "incluir edad y sexo como variables; «Desconocida» es categoría registrada, no
#              NaN → one-hot, no imputar" · §10 · Linfocitos % y Edad empatan en cabeza del IVF.
#
# ⚠️ DECISIÓN DE DISEÑO — `cxr_view` SE EXCLUYE (antes SÍ entraba, META_DIM era 18):
#     La proyección AP se hace al paciente encamado, así que es un PROXY DE GRAVEDAD, no información
#     biológica. Incluirla dejaría al modelo aprender el atajo "placa portátil ⇒ paciente grave" e
#     inflaría el resultado. Además la propia imagen ya contiene la proyección, con lo que la variable
#     es redundante para el backbone. Se conserva ÚNICAMENTE como eje de ESTRATIFICACIÓN en la
#     evaluación (bloque B8), que es la rama "monitorizar su efecto" del anexo del EDA.
#     ORIGEN: Anexo EDA (CXR) · "considerar cxr_view como covariable o al menos monitorizar su efecto
#     en Cardiomegalia" — se elige monitorizar, no incluir.
#
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#     si al estratificar por AP/PA el rendimiento difiere mucho (sobre todo en Cardiomegalia, que la
#     proyección AP magnifica), hay que discutirlo como SESGO DE ESPECTRO de la cohorte, y esta
#     exclusión es precisamente lo que impide que ese sesgo entre en el modelo.
# ══════════════════════════════════════════════════════════════════════════════
AGE_MIN, AGE_MAX = float(df_train["age"].min()), float(df_train["age"].max())
HRS_MIN, HRS_MAX = float(df_train["hours_adm_to_cxr"].min()), float(df_train["hours_adm_to_cxr"].max())

def build_metadata(df):
    N = len(df)
    M = np.zeros((N, META_DIM), dtype=np.float32)
    for i, (_, row) in enumerate(df.iterrows()):
        off = 0
        M[i, off] = (float(row.get("age", AGE_MIN)) - AGE_MIN) / (AGE_MAX - AGE_MIN + 1e-8); off += 1
        M[i, off] = float(GENDER_MAP.get(row.get("gender", 0), 0)); off += 1
        M[i, off + RACE_MAP.get(str(row.get("race", "UNKNOWN")).upper(), 0)] = 1.0; off += RACE_ONEHOT
        M[i, off + ADMISSION_MAP.get(str(row.get("admission_type", "EMERGENCY")).upper(), 1)] = 1.0; off += len(ADMISSION_MAP)
        M[i, off + ADM_LOC_MAP.get(str(row.get("admission_location", "EMERGENCY_ROOM")).upper(), 0)] = 1.0; off += len(ADM_LOC_MAP)
        M[i, off] = (float(row.get("hours_adm_to_cxr", HRS_MIN)) - HRS_MIN) / (HRS_MAX - HRS_MIN + 1e-8); off += 1
        # (cxr_view YA NO entra aquí: ver la nota de diseño de arriba)
    return M

meta_train = build_metadata(df_train)
meta_val   = build_metadata(df_val)
meta_test  = build_metadata(df_test)
assert meta_train.shape[1] == META_DIM, "META_DIM no coincide con las columnas realmente rellenadas"
print(f"Metadatos: train={meta_train.shape} val={meta_val.shape} test={meta_test.shape}  (cxr_view EXCLUIDA)")



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 7 · PREPROCESADO DE IMAGEN ALINEADO CON TORCHXRAYVISION (corrección A)║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Los .npy vienen normalizados z-score (estilo ImageNet, ~[-2.1, 2.6], 3 canales).
# torchxrayvision espera: 1 canal, rango [-1024, 1024], 224x224.
#
# Como el z-score es una transformación AFÍN, un min-max por imagen recupera la
# estructura [0,1] original sin necesidad de conocer la media/desv exactas usadas.
# Después escalamos a [-1024, 1024], que es exactamente lo que el backbone espera.

class CXRImageDataset(Dataset):
    # Devuelve tensores (1, 224, 224) listos para el backbone xrv.
    def __init__(self, df, cxr_npy):
        self.idx = df["_npy_idx"].to_numpy()
        self.cxr = cxr_npy

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        arr = np.asarray(self.cxr[int(self.idx[i])], dtype=np.float32)  # (3,320,320)
        g = arr[0] if arr.ndim == 3 else arr                            # 1 canal (gris replicado)
        gmin, gmax = float(g.min()), float(g.max())
        g = (g - gmin) / (gmax - gmin + 1e-8)                           # min-max -> [0,1]
        g = (2.0 * g - 1.0) * XRV_RANGE                                 # -> [-1024, 1024]
        t = torch.from_numpy(g)[None, None]                            # (1,1,H,W)
        t = F.interpolate(t, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        return t[0]                                                     # (1,224,224)

# Comprobación rápida del rango resultante (debe quedar cerca de [-1024, 1024]):
_sample = CXRImageDataset(df_train, cxr_train)[0]
print(f"Muestra preprocesada: shape={tuple(_sample.shape)} "
      f"min={_sample.min():.1f} max={_sample.max():.1f} (esperado ~[-1024, 1024])")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 8 · BACKBONE DenseNet121 + EXTRACCIÓN DE EMBEDDINGS (corrección B)    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Cargamos DenseNet121 preentrenado en TODAS las patologías de CheXpert/MIMIC.
# Lo usamos CONGELADO: solo extraemos el embedding de 1024 dims (avg-pool tras
# el último bloque denso), igual que hace su forward antes del clasificador.

def load_backbone():
    model = xrv.models.DenseNet(weights=BACKBONE_WEIGHTS)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model.to(DEVICE)

@torch.no_grad()
def extract_embeddings(model, df, cxr_npy, desc):
    loader = DataLoader(CXRImageDataset(df, cxr_npy), batch_size=EMB_BATCH,
                        shuffle=False, num_workers=0)
    out = []
    for x in tqdm(loader, desc=desc, leave=False):
        x = x.to(DEVICE)
        f = model.features(x)                       # (N,1024,7,7)
        f = F.relu(f, inplace=True)
        f = F.adaptive_avg_pool2d(f, (1, 1)).reshape(f.shape[0], -1)  # (N,1024)
        out.append(f.cpu().numpy())
    return np.concatenate(out, axis=0).astype(np.float32)

print("Backbone:", BACKBONE_WEIGHTS, "(se descargará la 1a vez)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 8b · CONTROL DE CALIDAD DE LA IMAGEN (salvaguarda del anexo del EDA)  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · control_calidad_cxr
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : recorre una muestra de radiografías (leídas del .npy con mmap, sin cargarlo entero)
#              y mide la desviación típica de píxel de cada una. Marca como DEGENERADA la que no
#              alcanza `umbral_std`, es decir, la que prácticamente no tiene contraste.
# POR QUÉ    : una imagen en blanco, negra o corrupta pasa silenciosamente por el backbone y produce
#              un embedding sin sentido que contamina el entrenamiento. Este chequeo lo detecta antes.
# ENTRADAS   : arr (np.memmap del .npy de imágenes) · n_muestra · umbral_std (0.15 sobre la escala
#              z-score original del .npy) · seed
# SALIDAS    : dict {n_revisadas, n_degeneradas, pct, std_min, indices_sospechosos}
# ORIGEN EDA : ANEXO (señal de imagen) · recuadro naranja "mantener el control de calidad (contraste
#              mínimo por imagen) como salvaguarda en inferencia". En el anexo salieron 0 degeneradas
#              sobre 1500 muestreadas, pero conviene dejar la comprobación activa.
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              si aparecen degeneradas, hay que EXCLUIRLAS (o regenerar el .npy) y decir cuántas eran:
#              en fases previas del proyecto hubo un problema de .npy corrupto que hundió el AUC a ~0,5.
# ══════════════════════════════════════════════════════════════════════════════
def control_calidad_cxr(arr, n_muestra=800, umbral_std=0.15, seed=SEED):
    rng = np.random.RandomState(seed)
    n = arr.shape[0]
    idx = np.sort(rng.choice(n, min(n_muestra, n), replace=False))
    stds = np.empty(len(idx), np.float32)
    for k, i in enumerate(idx):
        stds[k] = float(np.asarray(arr[int(i)], np.float32).std())
    mala = stds < umbral_std
    return {"n_revisadas": int(len(idx)), "n_degeneradas": int(mala.sum()),
            "pct": float(mala.mean() * 100), "std_min": float(stds.min()),
            "indices_sospechosos": [int(x) for x in idx[mala][:20]]}

_qc = control_calidad_cxr(cxr_train)
print(f"== Control de calidad de imagen (muestra de {_qc['n_revisadas']}) ==")
print(f"   Degeneradas (std < 0.15): {_qc['n_degeneradas']} ({_qc['pct']:.2f} %) · std minimo observado: {_qc['std_min']:.3f}")
if _qc["n_degeneradas"] > 0:
    print("   [AVISO] Hay imagenes sin contraste. Revisar/regenerar el .npy antes de extraer embeddings:")
    print("           indices sospechosos ->", _qc["indices_sospechosos"])
else:
    print("   OK: ninguna imagen degenerada (coincide con el anexo del EDA).")
json.dump(_qc, open(OUTPUT_DIR / "control_calidad_imagen.json", "w", encoding="utf-8"),
          indent=2, ensure_ascii=False)



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 9 · CALCULAR (O CARGAR) EMBEDDINGS — único paso lento (~30-45 min)    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Se cachean en disco. Si ya existen, se cargan en segundos (re-ejecución barata).

EMB_TRAIN_PATH = EMB_DIR / "emb_train.npy"
EMB_VAL_PATH   = EMB_DIR / "emb_val.npy"
EMB_TEST_PATH  = EMB_DIR / "emb_test.npy"

if EMB_TRAIN_PATH.exists() and EMB_VAL_PATH.exists() and EMB_TEST_PATH.exists():
    emb_train = np.load(EMB_TRAIN_PATH)
    emb_val   = np.load(EMB_VAL_PATH)
    emb_test  = np.load(EMB_TEST_PATH)
    print("Embeddings cargados desde caché.")
else:
    t0 = time.time()
    backbone = load_backbone()
    emb_train = extract_embeddings(backbone, df_train, cxr_train, "train")
    np.save(EMB_TRAIN_PATH, emb_train)
    emb_val = extract_embeddings(backbone, df_val, cxr_val, "val")
    np.save(EMB_VAL_PATH, emb_val)
    emb_test = extract_embeddings(backbone, df_test, cxr_test, "test")
    np.save(EMB_TEST_PATH, emb_test)
    del backbone; gc.collect()
    print(f"Embeddings calculados y cacheados en {(time.time()-t0)/60:.1f} min.")

EMB_DIM = emb_train.shape[1]
print(f"emb_train={emb_train.shape}  emb_val={emb_val.shape}  emb_test={emb_test.shape}  EMB_DIM={EMB_DIM}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 10 · CABEZA DE CLASIFICACIÓN + PÉRDIDA ENMASCARADA + UTILIDADES       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class LabelCorrelation(nn.Module):
    # Mezcla suave entre logits para modelar co-ocurrencia (Jaccard<=0.30 -> aporte modesto).
    def __init__(self, n=N_LABELS):
        super().__init__()
        self.lin = nn.Linear(n, n, bias=False)
        nn.init.eye_(self.lin.weight); self.lin.weight.data *= 0.1
        self.norm = nn.LayerNorm(n)
    def forward(self, x):
        return self.norm(x + self.lin(x))


class CXRHead(nn.Module):
    # Embedding (1024) + metadatos (13) -> fusión -> logits (6).
    def __init__(self, emb_dim=EMB_DIM, dropout=0.3, use_corr=True, use_meta=True):
        super().__init__()
        self.use_meta = use_meta
        self.img = nn.Sequential(nn.Linear(emb_dim, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True))
        if use_meta:
            self.meta = nn.Sequential(nn.Linear(META_DIM, 64), nn.BatchNorm1d(64), nn.ReLU(inplace=True))
            fused = 512 + 64
        else:
            self.meta = None; fused = 512
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(fused, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, N_LABELS),
        )
        self.corr = LabelCorrelation() if use_corr else None
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, emb, meta=None):
        z = self.img(emb)
        if self.use_meta and meta is not None:
            z = torch.cat([z, self.meta(meta)], dim=1)
        logits = self.head(z)
        if self.corr is not None:
            logits = self.corr(logits)
        return logits


class MaskedLoss(nn.Module):
    # BCE (o focal) con pos_weight por etiqueta y máscara de observabilidad.
    def __init__(self, pos_weight, loss_type="bce", focal_gamma=2.0):
        super().__init__()
        self.register_buffer("pos_weight", pos_weight)
        self.loss_type = loss_type
        self.gamma = focal_gamma
    def forward(self, logits, labels, mask):
        pw = self.pos_weight.to(logits.device)
        if self.loss_type == "focal":
            # focal: baja el peso de los ejemplos fáciles (útil en desequilibrio).
            bce = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pw, reduction="none")
            p = torch.sigmoid(logits)
            pt = labels * p + (1 - labels) * (1 - p)
            loss = ((1 - pt) ** self.gamma) * bce
        else:
            loss = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pw, reduction="none")
        loss = loss * mask
        return loss.sum() / mask.sum().clamp(min=1e-8)


def dynamic_pos_weights(y, m, clip=10.0):
    # pos_weight = n_neg / n_pos POR etiqueta, calculado sobre los objetivos EFECTIVOS
    # (ya con política de inciertos + negativos derivados aplicados). Es lo correcto:
    # compensa el desequilibrio que el modelo realmente ve.
    w = np.ones(N_LABELS, dtype=np.float32)
    for j in range(N_LABELS):
        sel = m[:, j] == 1
        pos = (y[sel, j] == 1).sum(); neg = (y[sel, j] == 0).sum()
        w[j] = np.clip(neg / max(pos, 1), 1.0 / clip, clip)
    return torch.tensor(w, dtype=torch.float32)


def sample_weights_by_rarity(y, m):
    # WeightedRandomSampler: da más probabilidad a combinaciones de patologías raras
    # (el EDA: solo 1.2% tiene las 5 a la vez). Mejora el aprendizaje de co-diagnósticos.
    combo = [tuple(int(v) for v in (y[i] * m[i]).astype(int)) for i in range(len(y))]
    from collections import Counter
    c = Counter(combo)
    w = np.array([1.0 / c[k] for k in combo], dtype=np.float64)
    return torch.tensor(w, dtype=torch.double)


def multilabel_metrics(probs, labels, mask, label_list=LABELS, thresholds=None):
    if thresholds is None:
        thresholds = {l: 0.5 for l in label_list}
    res = {}
    for j, lbl in enumerate(LABELS):
        sel = mask[:, j] == 1
        yt = labels[sel, j]; yp = probs[sel, j]
        npos = int(yt.sum()); nneg = int((1 - yt).sum())
        if npos >= 2 and nneg >= 2:
            auc = roc_auc_score(yt, yp); ap = average_precision_score(yt, yp)
        else:
            auc = float("nan"); ap = float("nan")
        thr = thresholds.get(lbl, 0.5)
        f1 = f1_score(yt, (yp >= thr).astype(float), zero_division=0)
        res[lbl] = {"AUC": auc, "AP": ap, "F1": f1, "n_pos": npos, "n_neg": nneg, "thr": thr,
                    "prevalencia": npos/max(npos+nneg,1)}
    def macro_of(group, key):
        vals = [res[l][key] for l in group if not np.isnan(res[l][key])]
        return float(np.mean(vals)) if vals else float("nan")
    res["macro_AUC_core"] = macro_of(CORE_LABELS, "AUC")      # secundaria
    res["macro_AUC_path"] = macro_of(PATHOLOGY_LABELS, "AUC")  # secundaria
    res["macro_AP_core"]  = macro_of(CORE_LABELS, "AP")       # PRIMARIA
    res["macro_AP_path"]  = macro_of(PATHOLOGY_LABELS, "AP")   # PRIMARIA
    return res


def best_thresholds_by_f1(probs, labels, mask, label_list=LABELS, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 37)
    thr = {}
    for j, lbl in enumerate(LABELS):
        sel = mask[:, j] == 1
        yt = labels[sel, j]; yp = probs[sel, j]
        if yt.sum() < 2:
            thr[lbl] = 0.5; continue
        best_f1, best_t = -1, 0.5
        for t in grid:
            f = f1_score(yt, (yp >= t).astype(float), zero_division=0)
            if f > best_f1: best_f1, best_t = f, t
        thr[lbl] = float(best_t)
    return thr

print("Cabeza, pérdida y utilidades definidas.")



In [ ]:
# CELDA 10b · KIT DE EVALUACIÓN CLÍNICA (B1, B2, B3, B4, B6, B8) — implementa los recuadros naranjas del EDA que faltaban
# NOTA: B5 (importancia MI/ANOVA) y B7 (dependencia de flags MNAR) NO APLICAN a esta modalidad.
# Motivo (§6 EDA): el ECG es una senal continua SIEMPRE presente: no hay variables tabulares cuya
# importancia medir ni patron de ausencia que vigilar. Esa informacion MNAR entra por el modulo
# tabular y se propaga a la fusion. Aqui si aplican B1, B2, B3, B4, B6 y B8.
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · bootstrap_ci_metric                                          [B4]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : remuestrea con reemplazo y recalcula la métrica, devolviendo el intervalo percentil,
#              POR ETIQUETA y para el MACRO de las 5 patologías.
# POR QUÉ    : con 464 pacientes en test, una diferencia entre modelos puede ser azar; el IC es lo
#              que permite afirmar (o no) que un modelo supera a otro.
# ENTRADAS   : probs (N,6) · labels (N,6) · mask (N,6) · metric "ap"|"auc" · n_boot · alpha
# SALIDAS    : dict {etiqueta:(lo,hi)} + clave "macro_path"
# ORIGEN EDA : §1 · "reportar SIEMPRE IC bootstrap por el tamaño reducido de val/test".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              si los IC del stacking y del promedio SE SOLAPAN, no afirmar superioridad del stacking.
# ══════════════════════════════════════════════════════════════════════════════
def bootstrap_ci_metric(probs, labels, mask, metric="ap", n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.RandomState(seed)
    scorer = average_precision_score if metric == "ap" else roc_auc_score
    out, macro_vals = {}, []
    for j, l in enumerate(LABELS):
        idx = np.where(mask[:, j] == 1)[0]; yt, yp = labels[idx, j], probs[idx, j]
        if int(yt.sum()) < 2 or int((1 - yt).sum()) < 2:
            out[l] = (float("nan"), float("nan")); continue
        vals = []
        for _ in range(n_boot):
            bs = rng.randint(0, len(idx), len(idx))
            if yt[bs].sum() < 1 or (1 - yt[bs]).sum() < 1: continue
            vals.append(scorer(yt[bs], yp[bs]))
        out[l] = (float(np.percentile(vals, 100*alpha/2)), float(np.percentile(vals, 100*(1-alpha/2)))) if vals else (float("nan"), float("nan"))
    for _ in range(n_boot):
        bs = rng.randint(0, len(probs), len(probs)); per = []
        for j, l in enumerate(LABELS):
            if l not in PATHOLOGY_LABELS: continue
            sel = mask[bs, j] == 1; yt, yp = labels[bs][sel, j], probs[bs][sel, j]
            if yt.sum() < 1 or (1 - yt).sum() < 1: continue
            per.append(scorer(yt, yp))
        if per: macro_vals.append(np.mean(per))
    out["macro_path"] = (float(np.percentile(macro_vals, 100*alpha/2)),
                         float(np.percentile(macro_vals, 100*(1-alpha/2)))) if macro_vals else (float("nan"), float("nan"))
    return out

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · operating_points                                             [B1]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : fija en VALIDACIÓN tres umbrales por etiqueta: "f1" (equilibrio), "cribado" (el más
#              alto que aún da Se>=sens_target) y "confirm" (el más bajo que aún da Sp>=spec_target).
# POR QUÉ    : un solo umbral no sirve en clínica. Cribar exige no perder enfermos; confirmar exige
#              no alarmar en falso. Son dos decisiones distintas sobre el mismo modelo.
# ENTRADAS   : probs/labels/mask de VALIDACIÓN · sens_target · spec_target
# SALIDAS    : dict {"f1"|"cribado"|"confirm": {etiqueta: umbral}}
# ORIGEN EDA : §11 · "fijar en VAL alta sensibilidad (cribado) y alta especificidad (confirmación).
#              Reportar Se/Sp/VPP/VPN".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              el VPP depende de la PREVALENCIA (13-36 %): un VPP modesto puede valer para cribar y
#              ser inservible para confirmar. Discutir cada punto por su consecuencia clínica.
# ══════════════════════════════════════════════════════════════════════════════
def operating_points(probs, labels, mask, sens_target=0.90, spec_target=0.90):
    grid = np.linspace(0.01, 0.99, 99); pts = {"f1": {}, "cribado": {}, "confirm": {}}
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        if yt.sum() < 2 or (1 - yt).sum() < 2:
            for k in pts: pts[k][l] = 0.5
            continue
        best_f1, thr_f1 = -1, 0.5; thr_sens, thr_spec = grid[0], grid[-1]
        for t in grid:
            pred = (yp >= t).astype(float)
            tp = ((pred == 1) & (yt == 1)).sum(); fn = ((pred == 0) & (yt == 1)).sum()
            tn = ((pred == 0) & (yt == 0)).sum(); fp = ((pred == 1) & (yt == 0)).sum()
            f1 = f1_score(yt, pred, zero_division=0)
            if f1 > best_f1: best_f1, thr_f1 = f1, t
            if tp/max(tp+fn, 1) >= sens_target: thr_sens = max(thr_sens, t)
            if tn/max(tn+fp, 1) >= spec_target: thr_spec = min(thr_spec, t)
        pts["f1"][l], pts["cribado"][l], pts["confirm"][l] = float(thr_f1), float(thr_sens), float(thr_spec)
    return pts

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · clinical_report                                              [B1]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : evalúa unos umbrales y devuelve Se, Sp, VPP, VPN y la confusión por etiqueta.
# POR QUÉ    : AUC y AP resumen el ranking, pero la decisión se toma en UN umbral; el clínico
#              necesita saber cuántos enfermos se escapan y cuántas alarmas falsas se generan.
# ENTRADAS   : probs/labels/mask (TEST) · thresholds {etiqueta: umbral} · punto (nombre)
# SALIDAS    : DataFrame (punto, etiqueta, umbral, Se, Sp, VPP, VPN, TP/TN/FP/FN, prevalencia)
# ORIGEN EDA : §11 "Reportar Se/Sp/VPP/VPN" · §3 (la prevalencia condiciona el VPP).
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              comparar FALSOS NEGATIVOS en cribado frente a FALSOS POSITIVOS en confirmación.
# ══════════════════════════════════════════════════════════════════════════════
def clinical_report(probs, labels, mask, thresholds, punto="f1"):
    rows = []
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        t = thresholds.get(l, 0.5); pred = (yp >= t).astype(float)
        tp = int(((pred == 1) & (yt == 1)).sum()); tn = int(((pred == 0) & (yt == 0)).sum())
        fp = int(((pred == 1) & (yt == 0)).sum()); fn = int(((pred == 0) & (yt == 1)).sum())
        rows.append({"punto": punto, "etiqueta": l, "umbral": round(t, 3),
                     "Se": tp/max(tp+fn,1), "Sp": tn/max(tn+fp,1), "VPP": tp/max(tp+fp,1), "VPN": tn/max(tn+fn,1),
                     "TP": tp, "TN": tn, "FP": fp, "FN": fn, "prevalencia": (tp+fn)/max(tp+tn+fp+fn,1)})
    return pd.DataFrame(rows)

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · calibration_report                                           [B2]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : Brier score por etiqueta + puntos de la curva de fiabilidad (10 bins por cuantiles).
# POR QUÉ    : la herramienta clínica muestra PROBABILIDADES; si no están calibradas, un 0,8 no
#              significa "80 % de estos pacientes lo tienen" y la cifra engaña al médico.
# ENTRADAS   : probs/labels/mask · n_bins
# SALIDAS    : (DataFrame Brier por etiqueta, dict {etiqueta:(frac_obs, media_pred)})
# ORIGEN EDA : §11 · "verificar con Brier score y curva de fiabilidad; recomprobar tras fusión".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              comparar Brier ANTES vs DESPUÉS; si no mejora, decirlo. RECALIBRAR tras la fusión.
# ══════════════════════════════════════════════════════════════════════════════
def calibration_report(probs, labels, mask, n_bins=10):
    rows, curves = [], {}
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        if len(np.unique(yt)) < 2:
            rows.append({"etiqueta": l, "Brier": float("nan")}); continue
        rows.append({"etiqueta": l, "Brier": float(brier_score_loss(yt, np.clip(yp, 0, 1)))})
        try: curves[l] = calibration_curve(yt, np.clip(yp, 0, 1), n_bins=n_bins, strategy="quantile")
        except Exception: curves[l] = (np.array([]), np.array([]))
    return pd.DataFrame(rows), curves

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · consistency_no_finding                                       [B6]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : correlación entre P("Sin hallazgo") y max P(patología), y % de casos en que ambas
#              superan 0,5 a la vez (el modelo se contradice).
# POR QUÉ    : las 6 cabezas son independientes; nada las obliga a ser coherentes. Un modelo que
#              afirma "sano" y "con derrame" a la vez es inaceptable en una herramienta clínica.
# ENTRADAS   : probs (N,6)
# SALIDAS    : dict {correlación (debe ser NEGATIVA), % incoherentes}
# ORIGEN EDA : §3 · "P(normal) útil como chequeo de consistencia, nunca como fuente de negativos".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              correlación ~0 o positiva ⇒ la cabeza "Sin hallazgo" no aprende normalidad.
# ══════════════════════════════════════════════════════════════════════════════
def consistency_no_finding(probs):
    j_nf = LABELS.index(NO_FINDING); j_p = [j for j in range(N_LABELS) if j != j_nf]
    p_nf, p_max = probs[:, j_nf], probs[:, j_p].max(axis=1)
    return {"corr_NoFinding_vs_maxPatologia": float(np.corrcoef(p_nf, p_max)[0, 1]),
            "pct_incoherentes": float(((p_nf > 0.5) & (p_max > 0.5)).mean()*100)}

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · stratified_report                                            [B8]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : recalcula la métrica primaria dentro de cada subgrupo (sexo, etnia, ingreso) y por
#              PROYECCIÓN radiográfica (cxr_view).
# POR QUÉ    : (a) equidad; (b) robustez — la placa AP se hace al paciente encamado y magnifica la
#              silueta cardíaca, así que conviene ver si el rendimiento depende de la proyección.
# ENTRADAS   : df (metadatos del split) · probs/labels/mask · cols
# SALIDAS    : DataFrame (variable, grupo, n, macro_AP, macro_AUC)
# ORIGEN EDA : §2/§9 "evaluar equidad por sexo y etnia" · ANEXO CXR "monitorizar cxr_view".
# DECISIÓN DE DISEÑO: cxr_view se usa SOLO aquí. NUNCA como predictor: es proxy de gravedad y su
#              inclusión inflaría el resultado por un atajo asistencial en lugar de señal biológica.
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              subgrupos con n<60 pueden diferir por PURO RUIDO; no afirmar inequidad sin IC.
# ══════════════════════════════════════════════════════════════════════════════
def stratified_report(df, probs, labels, mask, cols=("gender", "race", "admission_type", "cxr_view")):
    d = df.reset_index(drop=True); rows = []
    for col in cols:
        if col not in d.columns: continue
        for v in sorted(d[col].dropna().unique(), key=str):
            idx = d.index[d[col] == v].to_numpy()
            if len(idx) < 15:
                rows.append({"variable": col, "grupo": str(v), "n": len(idx), "macro_AP": np.nan, "macro_AUC": np.nan}); continue
            mm = multilabel_metrics(probs[idx], labels[idx], mask[idx])
            rows.append({"variable": col, "grupo": str(v), "n": len(idx),
                         "macro_AP": mm["macro_AP_path"], "macro_AUC": mm["macro_AUC_path"]})
    return pd.DataFrame(rows)



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 11 · BUCLE DE ENTRENAMIENTO DE LA CABEZA (sobre embeddings, rápido)   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def make_loader(emb, meta, y, m, cfg, train=True):
    ds = TensorDataset(torch.tensor(emb), torch.tensor(meta),
                       torch.tensor(y), torch.tensor(m))
    if train and cfg.get("use_sampler", False):
        sw = sample_weights_by_rarity(y, m)
        sampler = WeightedRandomSampler(sw, num_samples=len(sw), replacement=True)
        return DataLoader(ds, batch_size=HEAD_BATCH, sampler=sampler, drop_last=True)
    return DataLoader(ds, batch_size=HEAD_BATCH, shuffle=train, drop_last=train)


def train_head(emb_tr, meta_tr, y_tr, m_tr, emb_va, meta_va, y_va, m_va,
               cfg, n_epochs, patience, verbose=False):
    # pos_weight dinámico calculado SOBRE EL FOLD de entrenamiento actual.
    pw = dynamic_pos_weights(y_tr, m_tr)
    criterion = MaskedLoss(pw, loss_type=cfg.get("loss_type", "bce")).to(DEVICE)

    head = CXRHead(EMB_DIM, dropout=cfg["dropout"],
                   use_corr=cfg["use_corr"], use_meta=cfg["use_meta"]).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=1e-7)

    loader_tr = make_loader(emb_tr, meta_tr, y_tr, m_tr, cfg, train=True)
    best_auc, best_state, waited = -1.0, None, 0

    for ep in range(n_epochs):
        head.train()
        for eb, mb, yb, msb in loader_tr:
            eb, mb, yb, msb = eb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE), msb.to(DEVICE)
            opt.zero_grad()
            logits = head(eb, mb)
            loss = criterion(logits, yb, msb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
            opt.step()
        sched.step()

        probs_va = predict_head(head, emb_va, meta_va)
        mva = multilabel_metrics(probs_va, y_va, m_va)
        score = mva["macro_AP_path"]
        if verbose:
            print(f"    ep{ep+1:02d} macroAUC_path={score:.4f} core={mva['macro_AUC_core']:.4f}")
        if not np.isnan(score) and score > best_auc + 1e-4:
            best_auc, best_state, waited = score, copy.deepcopy(head.state_dict()), 0
        else:
            waited += 1
            if waited >= patience:
                break

    if best_state is not None:
        head.load_state_dict(best_state)
    return head, best_auc


@torch.no_grad()
def predict_head(head, emb, meta):
    head.eval()
    probs = []
    for i in range(0, len(emb), 512):
        eb = torch.tensor(emb[i:i+512]).to(DEVICE)
        mb = torch.tensor(meta[i:i+512]).to(DEVICE)
        probs.append(torch.sigmoid(head(eb, mb)).cpu().numpy())
    return np.concatenate(probs, axis=0)

print("Funciones de entrenamiento/predicción de la cabeza listas.")



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 12 · BÚSQUEDA DE HIPERPARÁMETROS CON OPTUNA (3-fold CV sobre train)   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Como entrenar la cabeza sobre embeddings cuesta segundos, podemos permitirnos
# una búsqueda TPE decente. Objetivo = macro_AUC_path medio en 3-fold CV interno.

def objective(trial):
    cfg = {
        "lr":           trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True),
        "dropout":      trial.suggest_float("dropout", 0.1, 0.5),
        "use_corr":     trial.suggest_categorical("use_corr", [True, False]),
        "use_meta":     trial.suggest_categorical("use_meta", [True, False]),
        "use_sampler":  trial.suggest_categorical("use_sampler", [True, False]),
        "loss_type":    trial.suggest_categorical("loss_type", ["bce", "focal"]),
        "policy":       "ignore",   # FIJA U-IGNORE (no se tunea)
    }
    y_all, m_all = build_targets(df_train, cfg["policy"])
    kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []
    for tr, va in kf.split(np.arange(len(df_train))):
        head, _ = train_head(
            emb_train[tr], meta_train[tr], y_all[tr], m_all[tr],
            emb_train[va], meta_train[va], y_all[va], m_all[va],
            cfg, TUNE_EPOCHS, TUNE_PATIENCE)
        p = predict_head(head, emb_train[va], meta_train[va])
        scores.append(multilabel_metrics(p, y_all[va], m_all[va])["macro_AP_path"])
        del head; gc.collect()
    return float(np.nanmean(scores))

t0 = time.time()
study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

BEST = dict(study.best_params); BEST["policy"] = "ignore"
print(f"\nBúsqueda terminada en {(time.time()-t0)/60:.1f} min")
print(f"Mejor macro_AUC_path (CV interno): {study.best_value:.4f}")
print("Mejor configuración:")
for k, v in BEST.items():
    print(f"   {k:14s} = {v}")



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 13 · K-FOLD FINAL -> OOF para stacking + predicciones test/val        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Con la mejor config entrenamos K modelos. Cada muestra de train recibe una
# predicción OOF (de un modelo que NO la vio -> sin fuga, válida para stacking).
# Las predicciones de val y test se promedian sobre los K modelos (ensemble).

cfg = dict(BEST)
y_train, m_train = build_targets(df_train, cfg["policy"])
y_val,   m_val   = build_targets(df_val,   cfg["policy"])
y_test,  m_test  = build_targets(df_test,  cfg["policy"])

oof_train = np.zeros((len(df_train), N_LABELS), dtype=np.float32)
acc_val   = np.zeros((len(df_val),   N_LABELS), dtype=np.float32)
acc_test  = np.zeros((len(df_test),  N_LABELS), dtype=np.float32)

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_metrics = []
t0 = time.time()
for k, (tr, va) in enumerate(kf.split(np.arange(len(df_train)))):
    head, best_auc = train_head(
        emb_train[tr], meta_train[tr], y_train[tr], m_train[tr],
        emb_train[va], meta_train[va], y_train[va], m_train[va],
        cfg, FINAL_EPOCHS, FINAL_PATIENCE)
    oof_train[va] = predict_head(head, emb_train[va], meta_train[va])
    acc_val  += predict_head(head, emb_val,  meta_val)
    acc_test += predict_head(head, emb_test, meta_test)
    mfold = multilabel_metrics(oof_train[va], y_train[va], m_train[va])
    fold_metrics.append(mfold)
    print(f"Fold {k+1}/{K_FOLDS}: macroAUC_path={mfold['macro_AP_path']:.4f} core={mfold['macro_AUC_core']:.4f}")
    del head; gc.collect()

val_pred  = acc_val  / K_FOLDS
test_pred = acc_test / K_FOLDS
print(f"\nK-fold terminado en {(time.time()-t0)/60:.1f} min")

# Métricas OOF globales (estimación honesta sobre todo el train)
oof_metrics = multilabel_metrics(oof_train, y_train, m_train)
print(f"OOF macro_AUC_path={oof_metrics['macro_AP_path']:.4f}  core={oof_metrics['macro_AUC_core']:.4f}")



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 14 · CALIBRACIÓN (VAL) + Brier (B2) + PUNTOS DE OPERACIÓN (B1) + PRE-REGISTRO (B3) ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# PROTOCOLO (§11 EDA): TRAIN entrena · VAL calibra, fija umbrales y selecciona · TEST se toca UNA vez.
# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · apply_calibration
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : aplica a cada columna de probabilidades el regresor isotónico aprendido en VALIDACIÓN.
# POR QUÉ    : la cabeza MLP sobre embeddings devuelve sigmoides bien ordenadas pero no calibradas.
#              Probabilidades fiables son imprescindibles para la herramienta clínica Y son mejores
#              features para el meta-modelo del stacking.
# ENTRADAS   : probs (N,6) probabilidades crudas
# SALIDAS    : (N,6) probabilidades calibradas
# ORIGEN EDA : §11 · "isotónica en VAL; verificar con Brier y curva de fiabilidad; recomprobar tras fusión".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              VAL son 750 pacientes; la isotónica puede sobreajustar. Si el Brier no mejora, decirlo.
# ══════════════════════════════════════════════════════════════════════════════
calibrators = {}
for j, lbl in enumerate(LABELS):
    sel = m_val[:, j] == 1
    yt = y_val[sel, j]; yp = val_pred[sel, j]
    if len(np.unique(yt)) < 2:
        calibrators[lbl] = None
        continue
    iso = IsotonicRegression(out_of_bounds="clip"); iso.fit(yp, yt); calibrators[lbl] = iso

def apply_calibration(probs):
    out = probs.copy()
    for j, lbl in enumerate(LABELS):
        if calibrators[lbl] is not None:
            out[:, j] = calibrators[lbl].predict(probs[:, j])
    return out

val_pred_cal  = apply_calibration(val_pred)
test_pred_cal = apply_calibration(test_pred)
oof_cal       = apply_calibration(oof_train)

# ── B2 · ¿mejora realmente la calibración? Brier antes vs después (medido en VAL) ─────────────
brier_pre, _       = calibration_report(val_pred,     y_val, m_val)
brier_post, curvas = calibration_report(val_pred_cal, y_val, m_val)
cal_cmp = brier_pre.merge(brier_post, on="etiqueta", suffixes=("_sin_calibrar", "_calibrado"))
cal_cmp["mejora"] = cal_cmp["Brier_sin_calibrar"] - cal_cmp["Brier_calibrado"]
print("== B2 · Brier en VALIDACION (menor = mejor) ==")
print(cal_cmp.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"   Mejora en {(cal_cmp['mejora'] > 0).sum()}/{len(cal_cmp)} etiquetas")
cal_cmp.to_csv(OUTPUT_DIR / "calibracion_brier.csv", index=False)

fig, ax = plt.subplots(figsize=(6.4, 6))
ax.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="calibracion perfecta")
for lbl in LABELS:
    fr, mp = curvas.get(lbl, (np.array([]), np.array([])))
    if len(fr): ax.plot(mp, fr, "o-", ms=4, lw=1.6, label=lbl)
ax.set_xlabel("probabilidad predicha media"); ax.set_ylabel("frecuencia observada")
ax.set_title("CXR v2 · Curva de fiabilidad tras calibracion (VAL)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "curva_fiabilidad.png", dpi=150, bbox_inches="tight"); plt.show()

# ── B1 · Tres puntos de operación fijados en VAL ──────────────────────────────────────────────
PUNTOS = operating_points(val_pred_cal, y_val, m_val, sens_target=0.90, spec_target=0.90)
thr_val = PUNTOS["f1"]
print("\n== B1 · Umbrales por punto de operacion (fijados en VAL) ==")
print("{:18s} {:>6} {:>18} {:>18}".format("Etiqueta", "F1", "Cribado(Se>=.90)", "Confirm(Sp>=.90)"))
for lbl in LABELS:
    print("{:18s} {:6.2f} {:18.2f} {:18.2f}".format(lbl, PUNTOS["f1"][lbl], PUNTOS["cribado"][lbl], PUNTOS["confirm"][lbl]))

# ── B3 · PRE-REGISTRO de la regla de decisión (ANTES de tocar TEST) ───────────────────────────
json.dump({"modelo": "CXR v2 optimo (DenseNet121 torchxrayvision congelado + cabeza MLP sobre embeddings)",
           "etiquetado": "POS=(==1); NEG=(==0)|(NaN->0); -1 ENMASCARADO (U-Ignore); sin derivar de No Finding",
           "metrica_primaria": "AUC-PR (macro_AP_path); ROC secundaria; IC bootstrap 1000",
           "calibracion": "isotonica ajustada en VAL",
           "imagen": "npy z-score -> min-max por imagen -> [-1024,1024] 1 canal 224x224 (rango que espera torchxrayvision)",
           "puntos_operacion": {"f1": PUNTOS["f1"], "cribado_Se>=0.90": PUNTOS["cribado"], "confirmacion_Sp>=0.90": PUNTOS["confirm"]},
           "cxr_view": "EXCLUIDA como predictor (proxy de gravedad); solo para estratificar",
           "nota": "TEST se evalua UNA sola vez con estos umbrales; no se reajusta nada despues."},
          open(OUTPUT_DIR / "preregistro_regla_decision.json", "w", encoding="utf-8"),
          indent=2, default=str, ensure_ascii=False)
print("\n== B3 · Regla PRE-REGISTRADA en preregistro_regla_decision.json (test aun sin tocar)")



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 15 · EVALUACIÓN EN TEST (UNA sola vez, con la regla PRE-REGISTRADA)   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Contenido: discriminación con IC bootstrap (B4) · 3 puntos de operación con Se/Sp/VPP/VPN (B1)
#            · consistencia de «Sin hallazgo» (B6) · estratificación por subgrupos y proyección (B8).

test_metrics = multilabel_metrics(test_pred_cal, y_test, m_test, thresholds=thr_val)
ci_ap  = bootstrap_ci_metric(test_pred_cal, y_test, m_test, metric="ap")
ci_auc = bootstrap_ci_metric(test_pred_cal, y_test, m_test, metric="auc")
rows = []
print("== DISCRIMINACION EN TEST (probabilidades calibradas) ==")
print("La AP es la metrica PRIMARIA; su linea base es la PREVALENCIA de cada etiqueta.")
print("{:18s} {:>7} {:>16} {:>6} {:>7} {:>16} {:>5} {:>5}".format(
    "Etiqueta", "AP", "IC95% AP", "prev", "AUC", "IC95% AUC", "N+", "N-"))
print("-" * 96)
for lbl in LABELS:
    m = test_metrics[lbl]
    auc = f"{m['AUC']:.4f}" if not np.isnan(m['AUC']) else "  N/A"
    la, ha = ci_ap[lbl]; lu, hu = ci_auc[lbl]
    fa = f"[{la:.3f},{ha:.3f}]" if not np.isnan(la) else "        N/A"
    fu = f"[{lu:.3f},{hu:.3f}]" if not np.isnan(lu) else "        N/A"
    print("{:18s} {:7.4f} {:>16} {:6.3f} {:>7} {:>16} {:5d} {:5d}".format(
        lbl, m["AP"], fa, m["prevalencia"], auc, fu, m["n_pos"], m["n_neg"]))
    rows.append({"label": lbl, **{k: m[k] for k in ["AUC", "AP", "F1", "n_pos", "n_neg", "prevalencia", "thr"]},
                 "AP_ci_lo": la, "AP_ci_hi": ha, "AUC_ci_lo": lu, "AUC_ci_hi": hu,
                 "AP_supera_prevalencia": bool(m["AP"] > m["prevalencia"])})
print("-" * 96)
mlo, mhi = ci_ap["macro_path"]; ulo, uhi = ci_auc["macro_path"]
print(f"MACRO patol.: AP={test_metrics['macro_AP_path']:.4f} IC95%=[{mlo:.3f},{mhi:.3f}]  (PRIMARIA)")
print(f"              AUC={test_metrics['macro_AUC_path']:.4f} IC95%=[{ulo:.3f},{uhi:.3f}]  (secundaria)")
n_ok = sum(r["AP_supera_prevalencia"] for r in rows if r["label"] in PATHOLOGY_LABELS)
print(f"Patologias con AP > prevalencia (= senal real): {n_ok}/{len(PATHOLOGY_LABELS)}")
print("NOTA: el EDA (§5) anticipa que la IMAGEN debe liderar Atelectasia y Opacidad pulmonar,")
print("      donde las analiticas rondan el azar. Si aqui tampoco se separan, revisar preprocesado.")
pd.DataFrame(rows).to_csv(OUTPUT_DIR / "metrics_per_label_v2.csv", index=False)

# ── B1 · Métricas clínicas en los tres puntos de operación ────────────────────────────────────
clin = pd.concat([clinical_report(test_pred_cal, y_test, m_test, PUNTOS[k], punto=nm)
                  for k, nm in [("f1", "f1"), ("cribado", "cribado_Se>=0.90"), ("confirm", "confirmacion_Sp>=0.90")]],
                 ignore_index=True)
print("\n== PUNTOS DE OPERACION (test) ==")
cab = "    {:18s} {:>5} {:>6} {:>6} {:>6} {:>6} {:>4} {:>4}".format("Etiqueta", "thr", "Se", "Sp", "VPP", "VPN", "FN", "FP")
for punto in clin["punto"].unique():
    print("\n  · Punto " + str(punto) + ":"); print(cab)
    for _, r in clin[clin["punto"] == punto].iterrows():
        print("    {:18s} {:5.2f} {:6.3f} {:6.3f} {:6.3f} {:6.3f} {:4d} {:4d}".format(
            r["etiqueta"], r["umbral"], r["Se"], r["Sp"], r["VPP"], r["VPN"], int(r["FN"]), int(r["FP"])))
clin.to_csv(OUTPUT_DIR / "puntos_operacion_test.csv", index=False)

# ── B6 · Coherencia interna de «Sin hallazgo» ─────────────────────────────────────────────────
cons = consistency_no_finding(test_pred_cal)
print(f"\n== B6 · corr(P(Sin hallazgo), max P(patologia))={cons['corr_NoFinding_vs_maxPatologia']:.3f} "
      f"(debe ser NEGATIVA) · incoherentes={cons['pct_incoherentes']:.1f} %")

# ── B8 · Equidad y robustez por subgrupo (incluye la proyección radiográfica) ─────────────────
strat = stratified_report(df_test, test_pred_cal, y_test, m_test)
print("\n== B8 · Rendimiento por subgrupo (n<60 => posible ruido, no inequidad) ==")
print(strat.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
strat.to_csv(OUTPUT_DIR / "estratificacion_subgrupos.csv", index=False)

json.dump({"version": "CXR v2 optimo (DenseNet121 + cabeza MLP sobre embeddings)",
           "primary_metric": "AUC-PR (macro_AP_path)",
           "test_macro_ap_path": test_metrics["macro_AP_path"], "test_macro_ap_ci": [mlo, mhi],
           "test_macro_auc_path": test_metrics["macro_AUC_path"], "test_macro_auc_ci": [ulo, uhi],
           "test_per_label": {l: test_metrics[l] for l in LABELS},
           "consistencia_no_finding": cons},
          open(OUTPUT_DIR / "summary_v2.json", "w", encoding="utf-8"), indent=2, default=str, ensure_ascii=False)
print("\nGuardados: metrics_per_label_v2.csv · puntos_operacion_test.csv · estratificacion_subgrupos.csv · summary_v2.json")



In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 16 · EXPORTAR FEATURES PARA EL STACKING (lo que consume el meta-modelo)║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Para cada split guardamos hadm_id + probabilidad por etiqueta, con prefijo cxr_.
# - train: probabilidades OOF (sin fuga) -> para ENTRENAR el meta-modelo.
# - val/test: probabilidades del ensemble K-fold (crudas y calibradas).

def save_predictions(df, probs_raw, probs_cal, name):
    cols = {"hadm_id": df["hadm_id"].to_numpy()}
    for j, lbl in enumerate(LABELS):
        key = lbl.replace(" ", "_")
        cols[f"cxr_{key}"] = probs_raw[:, j]
        if probs_cal is not None:
            cols[f"cxr_{key}_cal"] = probs_cal[:, j]
    out = pd.DataFrame(cols)
    path = OUTPUT_DIR / f"cxr_pred_{name}.csv"
    out.to_csv(path, index=False)
    print(f"   guardado {path}  ({out.shape[0]} filas, {out.shape[1]} cols)")
    return out

print("Exportando features de stacking:")
save_predictions(df_train, oof_train, oof_cal,        "oof_train")
save_predictions(df_val,   val_pred,  apply_calibration(val_pred), "val")
save_predictions(df_test,  test_pred, test_pred_cal,  "test")

# Resumen JSON con métricas + configuración elegida
cv_core=[m["macro_AUC_core"] for m in fold_metrics]; cv_path=[m["macro_AP_path"] for m in fold_metrics]
summary = {
    "version": "v5 DenseNet121 + exclusividad + calibración (stacking-ready)",
    "backbone": BACKBONE_WEIGHTS,
    "best_config": BEST,
    "derive_negatives": DERIVE_NEGATIVES_FROM_EXCLUSIVITY,
    "cv_macro_core_mean": float(np.nanmean(cv_core)), "cv_macro_core_std": float(np.nanstd(cv_core)),
    "cv_macro_path_mean": float(np.nanmean(cv_path)), "cv_macro_path_std": float(np.nanstd(cv_path)),
    "test_macro_core": test_metrics["macro_AUC_core"],
    "test_macro_path": test_metrics["macro_AUC_path"],
    "test_per_label": {l: test_metrics[l] for l in LABELS},
}
with open(OUTPUT_DIR / "summary_v2.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str, ensure_ascii=False)
print(f"   guardado {OUTPUT_DIR / 'summary_v2.json'}")




In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 17 · GRÁFICAS (AUC por etiqueta + curvas de calibración)             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# AUC por etiqueta en test
ax = axes[0]
aucs = [test_metrics[l]["AUC"] for l in LABELS]
colors = ["#c0392b" if (np.isnan(a) or a < 0.55) else "#2980b9" for a in aucs]
ax.bar(range(N_LABELS), [0 if np.isnan(a) else a for a in aucs], color=colors, alpha=0.8)
ax.axhline(0.5, color="gray", ls="--", label="azar")
ax.set_xticks(range(N_LABELS)); ax.set_xticklabels([l[:11] for l in LABELS], rotation=35, ha="right")
ax.set_ylim(0, 1); ax.set_ylabel("AUC-ROC (test)"); ax.set_title("AUC por etiqueta")
ax.legend()
for i, a in enumerate(aucs):
    if not np.isnan(a): ax.text(i, a + 0.01, f"{a:.3f}", ha="center", fontsize=8)

# Calibración (reliability) de las patologías core
ax = axes[1]
for lbl in CORE_LABELS:
    j = LABELS.index(lbl); sel = m_test[:, j] == 1
    yt = y_test[sel, j]; yp = test_pred_cal[sel, j]
    bins = np.linspace(0, 1, 9); idx = np.digitize(yp, bins) - 1
    xs, ys = [], []
    for b in range(len(bins) - 1):
        mb = idx == b
        if mb.sum() > 5:
            xs.append(yp[mb].mean()); ys.append(yt[mb].mean())
    ax.plot(xs, ys, "o-", label=lbl, alpha=0.8)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="perfecta")
ax.set_xlabel("prob. predicha"); ax.set_ylabel("frec. observada")
ax.set_title("Curvas de calibración (core)"); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "results_v5.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELDA 18 · EQUIDAD Y ROBUSTEZ POR SUBGRUPO -> ya cubierto por B8 (CELDA 15) ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# stratified_report calcula el macro AUC-PR por sexo, etnia, tipo de ingreso y PROYECCION (cxr_view)
# y exporta estratificacion_subgrupos.csv.
# ORIGEN EDA: 2/9 evaluar equidad por sexo y etnia · ANEXO CXR monitorizar el efecto de cxr_view.
# DECISION DE DISENO: cxr_view NO es predictor (proxy de gravedad); solo eje de estratificacion.
# INTERPRETACION FUTURA: la AP magnifica la silueta cardiaca; si Cardiomegalia rinde muy distinto
# entre AP y PA hay que discutirlo como sesgo de espectro. Subgrupos con n<60 pueden ser ruido.
print("Equidad por subgrupo: ver estratificacion_subgrupos.csv (CELDA 15, bloque B8).")
try:
    print(strat.to_string(index=False))
except NameError:
    print("   (ejecuta antes la CELDA 15)")



---
## ✅ Conclusión — Módulo CXR v2 (óptimo)

**DenseNet-121 (CheXNet) congelado + cabeza MLP** sobre embeddings, con metadatos clínicos auxiliares,
**Optuna optimizando AUC-PR**, K-fold para **OOF sin fuga**, pérdida **BCE enmascarada** con
`pos_weight` por etiqueta y calibración isotónica en VAL.

Adopta el **etiquetado FINAL**, la métrica **AUC-PR con IC bootstrap**, el **kit clínico B1, B2, B3,
B4, B6 y B8** y el **control de calidad de imagen** del anexo.

### 📌 Para la documentación posterior (recuadros naranjas a redactar con los resultados)
- **La imagen debe liderar Atelectasia y Opacidad pulmonar.** El EDA mostró que en esas dos etiquetas
  las analíticas rondan el azar (AUC≈0,50), así que la señal tiene que salir de aquí. Si el CXR
  tampoco las separa, el problema es de **preprocesado o etiquetado**, no de modalidad — y hay que
  investigarlo antes de seguir.
- **Control de calidad**: reportar cuántas imágenes degeneradas aparecieron. En el anexo salieron
  **0 de 1500**; si ahora aparecen, hay que excluirlas y decirlo (en fases previas del proyecto un
  `.npy` corrupto hundió el AUC a ~0,5).
- **v2 vs v1**: la diferencia mide **cuánto aporta la cabeza MLP + metadatos** frente a una simple
  LogReg sobre los mismos embeddings. Comparar con la misma métrica y mirar si los **IC se solapan**.
- **Puntos de operación**: es en imagen donde más sentido clínico tiene un punto de **alta
  sensibilidad para cribado**. Contrastar sus falsos positivos con los falsos negativos del punto de
  confirmación.
- **Calibración**: ¿mejora el Brier? Recordar **recalibrar tras la fusión**.
- **Consistencia**: la correlación entre P(*Sin hallazgo*) y max P(patología) debe ser **negativa**.
- **Proyección AP/PA**: la comprobación más importante de este módulo. La AP **magnifica la silueta
  cardíaca**; si Cardiomegalia rinde muy distinto entre AP y PA, discutirlo como **sesgo de espectro**
  (y es la justificación de haber excluido `cxr_view` como predictor).
- **Equidad**: subgrupos con n<60 ⇒ ruido, no inequidad. No concluir sin IC.

